# 1. What the Van Eck & Waltman data provide

From the Van Eck & Waltman paper and the Zenodo dataset description:

- `source.tsv` — one row per OpenAlex *source* (journal / series). This file includes a flag indicating whether the source is a **core source** (after the 7-step selection procedure).
- `work.tsv` — one row per OpenAlex *work* (publication). This file includes a flag indicating whether the work is a **core publication** (result of the same procedure).

Note: the authors ran the 7-step procedure across OpenAlex works (year ≥ 2000) and stored the final decisions in these TSV files. These are the `is_core_source` and `is_core_work` flags you will use to enrich your STRICT dataset.

# 2. Overview — goal and high-level steps

**Goal:** add two columns to `openalex_works_strict_q1_q3.csv`:

- `is_core_publication` (0/1 or False/True)
- `is_core_source` (0/1 or False/True)

High-level steps

1. Unzip and inspect the CWTS dataset (e.g., `core_openalex_2025aug.zip`) to find `work.tsv` and `source.tsv`.
2. Inspect the column names of `work.tsv` and `source.tsv` to locate the ID and core-flag fields (e.g., `work_id`, `is_core_work`, `source_id`, `is_core_source`).
3. Parse your STRICT CSV so you have:
   - the work URL/id (column `id`) and a numeric work id (e.g. `work_id_num`), and
   - the source URL (from `primary_location["source"]["id"]`) and a numeric source id (e.g. `source_id_num`).

4. Merge / join operations:
   - Left join STRICT works with `work.tsv` on numeric work id → add `is_core_work` / `is_core_publication`.
   - Left join STRICT works with `source.tsv` on numeric source id → add `is_core_source`.

5. For missing core flags after the join, treat them as non-core (fill missing with `0` / `False`).
6. Save the enriched STRICT CSV (for example: `openalex_works_strict_q1_q3_with_core_flags.csv`).

Implementation tips

- Convert OpenAlex URLs to numeric IDs before joining (example: remove the `https://openalex.org/W` prefix for works and extract the digits after the final `S` for sources).
- Use `.astype("Int64")` to allow integer columns with `NaN` (nullable integer type) when creating numeric IDs.
- After the join, map numeric core flags to human-friendly labels if desired (e.g., `core` / `noncore`).

### Inspect the CWTS TSV files

In [1]:
import pandas as pd
import ast
import numpy as np


In [2]:

# 1) Load a few rows from work.tsv
work_core = pd.read_csv("work.tsv", sep="\t", nrows=5)
print(work_core.columns)
work_core.head()


Index(['work_id', 'work_type', 'pub_year', 'source_id', 'doi', 'is_core_work'], dtype='object')


,work_id,work_type,pub_year,source_id,doi,is_core_work
0,9,article,2006,4306509860,NaN,0
1,15,book-chapter,2013,4306463937,10.1007/978-3-642-32197-9_8,0
2,23,article,2012,4306525036,NaN,0
3,79,article,2009,11973042,NaN,0
4,87,article,2010,4306519811,NaN,0


In [3]:

# 2) Load a few rows from source.tsv
source_core = pd.read_csv("source.tsv", sep="\t", nrows=5)
print(source_core.columns)
source_core.head()


Index(['source_id', 'source', 'source_type', 'issn_l', 'is_core_source',
       'n_works', 'n_core_works'],
      dtype='object')


,source_id,source,source_type,issn_l,is_core_source,n_works,n_core_works
0,61661,Journal of Prosthodontics,journal,1059-941X,1,3675,2667
1,81127,European Heart Journal Supplements,journal,1520-765X,1,6821,1644
2,134611,Andrologie,journal,1166-2654,1,585,19
3,138974,Foodservice Research International,journal,1524-8275,1,107,80
4,146206,Journal of Carcinogenesis,journal,1477-3163,1,294,264


### Loading and merging data

#### Load STRICT works and keep both URL + numeric IDs

In [4]:
# Load your STRICT works file
df = pd.read_csv("openalex_works_strict_q1_q3.csv")

# --- Extract source_id (URL) from primary_location, like you already did ---

def extract_source_id(pl_str):
    if pd.isna(pl_str):
        return np.nan
    try:
        pl = ast.literal_eval(pl_str)
        src = pl.get("source") or {}
        return src.get("id")   # e.g. 'https://openalex.org/S9267903'
    except Exception:
        return np.nan

df["source_id"] = df["primary_location"].apply(extract_source_id)

#### Create numeric IDs for work and source

In [5]:
# Work numeric ID: remove 'https://openalex.org/W' and keep the digits
df["work_id_num"] = (
    df["id"]
    .str.replace("https://openalex.org/W", "", regex=False)
    .astype("Int64")          # Int64 allows NaN
)

# Source numeric ID: remove everything except the digits after 'S'
df["source_id_num"] = (
    df["source_id"]
    .str.extract(r"S(\d+)$")[0]   # capture number after final 'S'
    .astype("Int64")
)

df[["id", "work_id_num", "source_id", "source_id_num"]].head()


,id,work_id_num,source_id,source_id_num
0,https://openalex.org/W4323655724,4323655724,https://openalex.org/S9267903,9267903
1,https://openalex.org/W3000065748,3000065748,https://openalex.org/S2505707916,2505707916
2,https://openalex.org/W4304943299,4304943299,https://openalex.org/S166722454,166722454
3,https://openalex.org/W4282940252,4282940252,https://openalex.org/S9692511,9692511
4,https://openalex.org/W3155263273,3155263273,https://openalex.org/S171267539,171267539


#### Load CWTS core tables and keep only what we need

In [6]:
# Load all rows from work.tsv and source.tsv
work_core = pd.read_csv("work.tsv", sep="\t")
source_core = pd.read_csv("source.tsv", sep="\t")

# Ensure IDs are numeric (same dtype as our *_num columns)
work_core["work_id"] = work_core["work_id"].astype("Int64")
source_core["source_id"] = source_core["source_id"].astype("Int64")

# Keep only ID + core flag
work_core = work_core[["work_id", "is_core_work"]].copy()
source_core = source_core[["source_id", "is_core_source"]].copy()


#### Merge core publication flag onto STRICT

In [7]:
# Left join on numeric work_id
df = df.merge(
    work_core,
    how="left",
    left_on="work_id_num",
    right_on="work_id"
)

# Drop duplicate key column from CWTS table
df.drop(columns=["work_id"], inplace=True)

# Missing values: treat as non-core (0)
df["is_core_work"] = df["is_core_work"].fillna(0).astype(int)

# Optional human-friendly label
df["core_pub_status"] = df["is_core_work"].map({1: "core", 0: "noncore"})


#### Merge core source flag onto STRICT

In [8]:
df = df.merge(
    source_core,
    how="left",
    left_on="source_id_num",
    right_on="source_id",
    suffixes=("", "_core")
)

df.drop(columns=["source_id_core"], inplace=True)   # drop CWTS key, keep your URL column

df["is_core_source"] = df["is_core_source"].fillna(0).astype(int)
df["core_source_status"] = df["is_core_source"].map({1: "core", 0: "noncore"})


### Quick sanity checks

#### Check core works

In [9]:
print("Core vs non-core works:")
print(df["is_core_work"].value_counts())
print()

print("Core vs non-core works (share):")
print((df["is_core_work"].value_counts(normalize=True) * 100).round(2))


Core vs non-core works:
is_core_work
1    4359
0     705
Name: count, dtype: int64

Core vs non-core works (share):
is_core_work
1    86.08
0    13.92
Name: proportion, dtype: float64


#### Check core sources

In [10]:
print("Core vs non-core sources (per work row):")
print(df["is_core_source"].value_counts())
print()

print("Core vs non-core sources (share):")
print((df["is_core_source"].value_counts(normalize=True) * 100).round(2))


Core vs non-core sources (per work row):
is_core_source
1    4761
0     303
Name: count, dtype: int64

Core vs non-core sources (share):
is_core_source
1    94.02
0     5.98
Name: proportion, dtype: float64


#### Peek at a few rows to see if things line up

In [11]:
df[[
    "id",
    "work_id_num",
    "is_core_work",
    "core_pub_status",
    "source_id",
    "source_id_num",
    "is_core_source",
    "core_source_status"
]].head()


,id,work_id_num,is_core_work,core_pub_status,source_id,source_id_num,is_core_source,core_source_status
0,https://openalex.org/W4323655724,4323655724,1,core,https://openalex.org/S9267903,9267903,1,core
1,https://openalex.org/W3000065748,3000065748,1,core,https://openalex.org/S2505707916,2505707916,1,core
2,https://openalex.org/W4304943299,4304943299,1,core,https://openalex.org/S166722454,166722454,1,core
3,https://openalex.org/W4282940252,4282940252,1,core,https://openalex.org/S9692511,9692511,1,core
4,https://openalex.org/W3155263273,3155263273,1,core,https://openalex.org/S171267539,171267539,1,core


#### Save the enriched STRICT file

In [12]:
output_path = "openalex_works_strict_q1_q3_with_core_flags.csv"
df.to_csv(output_path, index=False)
print("Saved:", output_path)


Saved: openalex_works_strict_q1_q3_with_core_flags.csv


In [13]:
df.head()

,id,doi,title,abstract_inverted_index,publication_year,publication_date,open_access,type,language,cited_by_count,...,keep_q1_q3_any,is_strict_in_scopus,is_strict_q1_q3,source_id,work_id_num,source_id_num,is_core_work,core_pub_status,is_core_source,core_source_status
0,https://openalex.org/W4323655724,https://doi.org/10.1016/j.lindif.2023.102274,ChatGPT for good? On opportunities and challen...,NaN,2023,2023-03-09,"{'is_oa': True, 'oa_status': 'green', 'oa_url'...",article,en,3628,...,True,True,True,https://openalex.org/S9267903,4323655724,9267903,1,core,1,core
1,https://openalex.org/W3000065748,https://doi.org/10.1002/widm.1355,Educational data mining and learning analytics...,"{'Abstract': [0], 'This': [1, 93, 140], 'surve...",2020,2020-01-13,"{'is_oa': True, 'oa_status': 'green', 'oa_url'...",article,en,827,...,True,True,True,https://openalex.org/S2505707916,3000065748,2505707916,1,core,1,core
2,https://openalex.org/W4304943299,https://doi.org/10.1007/s10639-022-11316-w,Ethical principles for artificial intelligence...,NaN,2022,2022-10-13,"{'is_oa': True, 'oa_status': 'hybrid', 'oa_url...",article,en,795,...,True,True,True,https://openalex.org/S166722454,4304943299,166722454,1,core,1,core
3,https://openalex.org/W4282940252,https://doi.org/10.3389/fpsyg.2022.813632,Lessons Learned and Future Directions of MetaT...,"{'Self-regulated': [0], 'learning': [1, 6, 35,...",2022,2022-06-14,"{'is_oa': True, 'oa_status': 'gold', 'oa_url':...",review,en,144,...,True,True,True,https://openalex.org/S9692511,4282940252,9692511,1,core,1,core
4,https://openalex.org/W3155263273,https://doi.org/10.1007/s40593-021-00239-1,Ethics of AI in Education: Towards a Community...,"{'Abstract': [0], 'While': [1], 'Artificial': ...",2021,2021-04-09,"{'is_oa': True, 'oa_status': 'hybrid', 'oa_url...",article,en,770,...,True,True,True,https://openalex.org/S171267539,3155263273,171267539,1,core,1,core


### Spliting Core/Non-core sources and works

In [14]:
df = pd.read_csv("openalex_works_strict_q1_q3_with_core_flags.csv")

In [15]:
# Core works (is_core_work == 1)
core_works = df[df["is_core_work"] == 1].copy()

# Non-core works (is_core_work == 0)
noncore_works = df[df["is_core_work"] == 0].copy()

# Save
core_works.to_csv("openalex_STRICT_core_works.csv", index=False)
noncore_works.to_csv("openalex_STRICT_noncore_works.csv", index=False)

print("Core works:", core_works.shape)
print("Non-core works:", noncore_works.shape)


Core works: (4359, 48)
Non-core works: (705, 48)


In [16]:
# Works that appear in core sources (is_core_source == 1)
core_source_works = df[df["is_core_source"] == 1].copy()

# Works that appear in non-core sources (is_core_source == 0)
noncore_source_works = df[df["is_core_source"] == 0].copy()

# Save
core_source_works.to_csv("openalex_STRICT_core_sources_works.csv", index=False)
noncore_source_works.to_csv("openalex_STRICT_noncore_sources_works.csv", index=False)

print("Works in core sources:", core_source_works.shape)
print("Works in non-core sources:", noncore_source_works.shape)


Works in core sources: (4761, 48)
Works in non-core sources: (303, 48)
